**Data preparation**

* This file creates climate.db, which is used for accessing the data for the website.
* This document accesses the files in the data folder and creates a database.

In [1]:
__author__ = 'Maximilian Koch'

In [2]:
import pandas as pd
import numpy as np

In [3]:
#European countries excluding some tiny nations
countries = ['Albania','Andorra','Austria','Belarus','Belgium','Bosnia and Herzegovina',
           'Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France',
           'Germany','Greece','Hungary','Iceland','Ireland','Italy','Latvia','Liechtenstein',
           'Lithuania','Luxembourg','Malta','Moldova','Montenegro','Netherlands',
           'North Macedonia','Norway','Poland','Portugal','Romania','Russia','Serbia',
           'Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom']

#Corresponding ISO3 codes
ISO3 = ['ALB','AND','AUT','BLR','BEL','BIH','BGR',
'HRV','CYP','CZE','DNK','EST','FIN','FRA',
'DEU','GRC','HUN','ISL','IRL','ITA','LVA',
'LIE','LTU','LUX','MLT','MDA','MNE','NLD',
'MKD','NOR','POL','PRT','ROU','RUS','SRB',
'SVK','SVN','ESP','SWE','CHE','UKR','GBR']

All the following tables are formatted into the index containing the ISO3 countries and the row column containing the year.

**Statistic 1 - Annual Surface Temperature Change in comparison to 1951-1980, measured in °C**

In [4]:
ast = pd.read_csv('data/1 - Annual_Surface_Temperature_Change.csv')
europe = ast[ast['ISO3'].isin(ISO3)]
SURFACE_TEMP = europe.loc[:, 'F1961':'F2022'].set_index(europe['ISO3'])
SURFACE_TEMP.columns = SURFACE_TEMP.columns.str.replace('F','')
SURFACE_TEMP.columns = SURFACE_TEMP.columns.map(np.int64)

In [5]:
SURFACE_TEMP.head()

,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
ISO3,,,,,,,,,,,,,,,,,,,,,
ALB,0.627,0.326,0.075,-0.166,-0.388,0.559,-0.074,0.081,-0.013,-0.106,...,1.333,1.198,1.569,1.464,1.121,2.028,1.675,1.498,1.536,1.518
AND,0.736,0.112,-0.752,0.308,-0.490,0.415,0.637,0.018,-0.137,0.121,...,0.831,1.946,1.690,1.990,1.925,1.919,1.964,2.562,1.533,3.243
AUT,1.031,-0.621,-0.727,-0.371,-0.883,0.602,0.676,0.211,-0.126,-0.550,...,1.098,2.409,2.167,2.096,1.741,2.524,2.370,2.315,1.395,2.498
BLR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.212,2.180,2.247,2.201,1.592,2.342,2.689,3.510,1.728,1.922
BEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.463,2.417,1.371,1.882,1.731,2.196,2.081,2.572,1.203,2.807


**Statistic 2 - Renewable Energy generation in Gigawatt hours**

In [6]:
#Statistic 2
#Renewable Energy
re = pd.read_csv('data/2 - Renewable_Energy.csv')

In [7]:
europe = re[re['ISO3'].isin(ISO3)]
renewable = europe[(europe['Energy Type'] == 'Total Renewable') & 
                    (europe['Indicator'] == 'Electricity Generation')]
#Numbers for 2022 are not accurate (yet)
renewable = renewable.drop(columns=['Source','2022'])
RENEWABLE_ENERGY = renewable.groupby('ISO3').sum(numeric_only=True).T
RENEWABLE_ENERGY.index = RENEWABLE_ENERGY.index.map(np.int64)
RENEWABLE_ENERGY = RENEWABLE_ENERGY.T

In [8]:
RENEWABLE_ENERGY.head()

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
ISO3,,,,,,,,,,,,,,,,,,,,,
ALB,4594.000,3555.136,3512.272,4885.272,5466.272,5373.272,5431.272,2788.272,3797.272,5201.408,...,4725.924,6960.030,4725.629,5896.428,7783.062,4526.179,8553.486,5206.043,5313.166,8962.710
AND,80.000,80.000,80.000,80.000,94.000,85.000,74.000,76.000,78.700,81.000,...,87.858,114.683,127.007,99.722,98.777,102.607,135.430,158.884,142.169,96.983
AUT,43434.634,42232.465,41894.815,35312.928,39772.315,40884.588,40668.768,43177.482,44530.799,47177.243,...,51327.827,50491.402,50186.269,47582.040,51082.238,51055.166,50053.507,54633.638,55423.576,52753.663
BEL,1044.000,1075.000,1138.000,1192.000,1496.016,2106.340,2952.218,3486.154,4416.982,5439.008,...,10521.600,11731.400,12219.100,14460.600,14260.100,15810.300,17188.700,19483.500,23466.300,22724.700
BGR,2631.000,1651.000,2116.000,2995.000,3139.811,4302.015,4233.334,2876.709,2930.635,3680.262,...,5243.230,6912.119,7389.053,8759.958,7046.288,6131.118,9380.455,7484.584,7465.590,10308.256


**Statistic 3 - Number of Climate Policies**

In [9]:
policies = pd.read_csv('data/3 - Policy numbers.csv')

In [10]:
POLICY_NUMBER = pd.DataFrame(index=list(range(1990,2022)))
oecd_countries = set(policies['REF_AREA'])
#Filter for European countries and sum up different categories of policies
for country in ISO3:
    if country in oecd_countries:
        POLICY_NUMBER[country] = policies[policies['REF_AREA']==country].groupby('TIME_PERIOD').sum()['OBS_VALUE']
POLICY_NUMBER = POLICY_NUMBER.T

In [11]:
POLICY_NUMBER.head()

,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
AUT,12.0,12.0,14.0,14.0,16.0,22.0,28.0,28.0,28.0,30.0,...,66.0,68.0,70.0,72.0,76.0,76.0,76.0,80.0,80.0,84.0
BEL,4.0,10.0,10.0,10.0,14.0,18.0,20.0,20.0,20.0,22.0,...,64.0,62.0,68.0,68.0,68.0,70.0,70.0,72.0,74.0,82.0
BGR,2.0,2.0,2.0,2.0,4.0,4.0,4.0,4.0,4.0,4.0,...,42.0,44.0,46.0,48.0,50.0,50.0,50.0,52.0,52.0,56.0
HRV,0.0,0.0,2.0,2.0,4.0,6.0,6.0,6.0,6.0,6.0,...,18.0,38.0,40.0,42.0,40.0,42.0,44.0,46.0,46.0,54.0
CZE,2.0,2.0,2.0,6.0,8.0,12.0,14.0,14.0,14.0,14.0,...,60.0,62.0,64.0,64.0,66.0,68.0,66.0,68.0,68.0,70.0


**Statistic 4 - Fossil fuel emissions from 1850 in MtCO2**

In [12]:
emissions = pd.read_excel('data/4 - Fossil Fuel Emissions.xlsx', sheet_name='Territorial Emissions')

In [13]:
# Remove head and set year as index
emissions = emissions.set_axis(emissions.iloc[10], axis=1)
emissions = emissions.drop(emissions.index[:11])
emissions.columns.values[0] = 'Year'
emissions=emissions.set_index('Year')

In [14]:
FOSSIL_FUEL = emissions[countries]
FOSSIL_FUEL.columns = [ISO3[i] for i in range(len(countries))]
FOSSIL_FUEL = FOSSIL_FUEL.T
FOSSIL_FUEL *= 3.664 #Convert tonnes of carbon to tonnes of CO2 (more common unit)

In [15]:
FOSSIL_FUEL.head()

Year,1850,1851,1852,1853,1854,1855,1856,1857,1858,1859,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
ALB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.287465,5.99965,4.712144,4.631979,5.293048,4.894953,4.826944,5.018921,4.903652,4.95473
AND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.47632,0.461664,0.465328,0.468992,0.465328,0.49464,0.483648,0.373728,0.363046,0.368645
AUT,2.32664,2.333968,2.810288,3.22432,3.176688,3.70064,4.23192,4.87312,7.243728,5.866064,...,67.776042,64.1759,66.365641,67.226575,69.608667,66.571892,67.956158,62.121251,66.018632,61.488421
BLR,0.003167,0.0,0.0,0.0,0.0,0.009604,0.0,0.0,0.013486,0.014304,...,64.125556,63.648874,58.798911,58.134483,59.381853,62.156932,62.096344,59.055551,60.938197,58.801129
BEL,9.332208,10.1676,11.409696,11.787088,13.058496,13.377264,13.212384,13.754656,14.450816,14.897824,...,102.714747,97.029247,101.145633,99.623026,99.05504,99.967324,99.47026,91.101387,95.668081,89.605362


**Statistic 5 - Climate-related disaster frequency**

In [16]:
disasters = pd.read_csv('data/5 - Disaster Frequency.csv')

In [17]:
#Get number of total disasters of European countries
europe_disasters = disasters[(disasters['ISO3'].isin(ISO3)) &
(disasters['Indicator'] == 'Climate related disasters frequency, Number of Disasters: TOTAL')]
DISASTER_FREQUENCY = europe_disasters.loc[:, '1980':'2022'].set_index(europe_disasters['ISO3'])

In [18]:
DISASTER_FREQUENCY.head()

,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
ISO3,,,,,,,,,,,,,,,,,,,,,
ALB,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,1.0,...,NaN,NaN,3.0,1.0,2.0,1.0,NaN,NaN,1.0,NaN
AUT,1.0,1.0,2.0,1.0,2.0,1.0,NaN,NaN,NaN,NaN,...,1.0,NaN,NaN,1.0,2.0,NaN,2.0,NaN,1.0,1.0
BLR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN
BEL,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,NaN,NaN,...,2.0,1.0,1.0,2.0,NaN,2.0,3.0,2.0,3.0,1.0
BIH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4.0,1.0,NaN,NaN,NaN,2.0,1.0,1.0,1.0


**Statistic 6 - Emissions in tonnes of CO2-equivalent**

In [19]:
emissions = pd.read_csv('data/6 - Pollutions.csv')

In [20]:
EMISSIONS = pd.DataFrame(index=list(range(1988,2022)))
oecd_countries = set(policies['REF_AREA'])
#Filter for the European countries and reindex due to missing values
for country in ISO3:
    if country in oecd_countries:
        country_emissions = emissions[emissions['REF_AREA']==country]['OBS_VALUE']
        country_emissions.index = emissions[emissions['REF_AREA']==country]['TIME_PERIOD']
        EMISSIONS[country] = country_emissions.reindex(EMISSIONS.index, fill_value=pd.NA)
EMISSIONS = EMISSIONS.T

In [21]:
EMISSIONS.head()

,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
AUT,NaN,NaN,66839.80,63510.44,66620.30,59455.65,67054.99,60182.37,64257.04,60040.83,...,74020.84,73986.21,69050.39,72321.37,72828.52,78883.08,83775.26,82126.61,68688.65,67130.65
BEL,NaN,NaN,142908.44,146108.99,145787.34,144589.81,149390.27,151295.95,154747.82,146791.75,...,120167.36,119599.79,113971.73,118134.15,116651.14,116263.92,116976.66,115983.24,106937.75,110627.36
BGR,NaN,NaN,82566.08,64426.60,59373.53,58373.84,54189.30,55640.26,56673.12,53209.73,...,51654.63,47567.24,49365.03,52732.30,48285.83,50289.83,45798.19,44576.57,38579.08,44773.18
HRV,NaN,NaN,25141.98,17335.04,15350.31,15181.90,13997.31,14181.73,15029.98,17001.31,...,20932.23,18742.36,18151.10,18912.65,19125.45,20813.57,19097.16,19063.55,18240.03,18644.05
CZE,NaN,NaN,190189.53,170924.74,164965.26,157636.45,150027.71,148694.71,152296.51,148736.09,...,126916.17,122170.09,120045.67,121542.35,123999.68,126453.11,130181.88,131229.33,124339.90,126739.70


**Statistic 7 - Protected Land Areas in percentage of overall land area**

In [22]:
areas = pd.read_csv('data/7- Protected Areas.csv')

In [23]:
AREAS = pd.DataFrame(index=list(range(1950,2022)))
oecd_countries = set(policies['REF_AREA'])
#Filter for the European countries and reindex due to missing values
for country in ISO3:
    if country in oecd_countries:
        country_areas = areas[areas['REF_AREA']==country]['OBS_VALUE']
        country_areas.index = areas[areas['REF_AREA']==country]['TIME_PERIOD']
        AREAS[country] = country_areas.reindex(AREAS.index, fill_value=pd.NA)
AREAS = AREAS.T

In [24]:
AREAS.head()

,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
AUT,0.649277,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26.918779,27.141763,27.301846,27.539140,28.042643,28.043537,28.163674,28.852004,29.130436,29.385317
BEL,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.213062,7.235909,8.592030,9.070181,14.694575,14.711710,14.765563,15.359580,15.393034,15.473814
BGR,0.706895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40.545222,40.562337,40.562337,40.562562,40.685745,40.685970,40.744071,40.866803,40.867029,40.867254
HRV,0.817051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.270906,32.262612,37.436972,37.436972,37.446322,37.446322,37.446322,37.523352,37.523352,37.672959
CZE,0.146012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,19.904635,19.982720,20.061122,20.529631,20.710559,21.284133,21.705346,21.729470,21.753594,21.754546


**Statistic 8 - Droughts: cropland soil moisture change (%) compared to the baseline climatology (1981-2010)**

In [25]:
droughts = pd.read_csv('data/8 - Droughts.csv')

In [26]:
DROUGHTS = pd.DataFrame(index=list(range(1981,2023)))
oecd_countries = set(policies['REF_AREA'])
#Filter for the European countries and reindex due to missing values
for country in ISO3:
    if country in oecd_countries:
        country_droughts = droughts[droughts['REF_AREA']==country]['OBS_VALUE']
        country_droughts.index = droughts[droughts['REF_AREA']==country]['TIME_PERIOD']
        DROUGHTS[country] = country_droughts.reindex(DROUGHTS.index, fill_value=pd.NA)
DROUGHTS = DROUGHTS.T

In [27]:
DROUGHTS.head()

,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
AUT,-1.055157,-0.646071,-4.503289,-0.722368,-0.029970,-3.252666,3.742095,-1.415428,1.230446,-2.731682,...,1.474625,3.329665,-4.021475,2.850226,-3.591927,-3.657970,-2.550463,0.127346,-0.219743,-1.030414
BEL,7.551580,0.899769,-2.982519,3.125614,3.005828,-0.119551,7.493567,1.366266,-8.466669,-6.965235,...,-1.439715,2.181885,-3.168065,-1.118951,-3.893662,-8.867096,-4.577619,-10.966359,4.993123,-7.957756
BGR,0.338458,6.463355,0.148541,-1.702359,-4.103679,-3.855555,2.387827,2.056101,-1.681302,-7.512972,...,-2.679109,10.084460,1.372044,-0.203504,-1.085484,-1.814716,-5.533767,-6.984379,0.870997,-8.980108
HRV,2.969690,2.067686,-6.037046,1.945147,-3.155981,-0.295101,-0.632353,-1.978222,2.811210,-4.869891,...,-0.495270,7.940019,-3.034337,1.375875,-3.801120,-1.174056,0.073460,-4.233699,-3.489763,-5.380498
CZE,4.340863,-3.576201,-6.161205,4.234801,1.634343,-0.298039,6.316899,-0.964409,0.911089,-4.722371,...,2.278598,0.193338,-6.466067,-0.184922,-0.805942,-10.694443,-3.894401,0.536671,3.012280,0.116226


**Database creation**

In [28]:
import sqlite3

In [29]:
statistics = ['SURFACE_TEMP', 'RENEWABLE_ENERGY', 'POLICY_NUMBER', 'FOSSIL_FUEL',
              'DISASTER_FREQUENCY', 'EMISSIONS', 'AREAS', 'DROUGHTS']

In [30]:
COUNTRY_ID = pd.DataFrame({'country_names' : countries, 'ISO3' : ISO3})

In [31]:
SQL_TABLES = {}

In [32]:
#Turning the separate statistics all into one dataframe
#and one line for each data point instead of table
for stat in statistics:
    obj = eval(stat)
    df = obj.reset_index()
    df.rename(columns={'index' : 'ISO3'}, inplace=True)
    melted_df = df.melt(id_vars=['ISO3'], var_name='Year', value_name='Value')
    SQL_TABLES.update({stat : melted_df})

#Add name of statistic to each line
for stat in statistics:
    SQL_TABLES[stat]['stat'] = [stat]*len(SQL_TABLES[stat])
STATISTIC = pd.concat(list(SQL_TABLES.values()), ignore_index=True)

In [33]:
STATISTIC.head()

,ISO3,Year,Value,stat
0,ALB,1961,0.627,SURFACE_TEMP
1,AND,1961,0.736,SURFACE_TEMP
2,AUT,1961,1.031,SURFACE_TEMP
3,BLR,1961,NaN,SURFACE_TEMP
4,BEL,1961,NaN,SURFACE_TEMP


In [34]:
conn = sqlite3.connect('climate.db')

In [35]:
conn.execute('''CREATE TABLE countries
            (country_names CHAR PRIMARY KEY,
            ISO3 CHAR)''')
COUNTRY_ID.to_sql('countries', conn, if_exists='replace', index=False)

42

In [36]:
conn.execute('''CREATE TABLE statistics
            (ISO3 CHAR,
            Year REAL,
            Value REAL,
            Statistic CHAR)''')
STATISTIC.to_sql('statistics', conn, if_exists='replace', index=False)

18029

In [48]:
conn.close()